# 3단계: BPE(Byte Pair Encoding) 토크나이저

## 이 노트북에서 배우는 것

지금까지 두 가지 방법의 한계를 경험했습니다:

| 방법 | 장점 | 단점 |
|------|------|------|
| 글자 단위 | OOV 없음 | 토큰 수 너무 많음 |
| 단어 단위 | 토큰 수 적음 | OOV 빈번, vocab 폭발 |

**BPE(Byte Pair Encoding)** 는 두 방법의 장점만 가져옵니다:

- **글자(바이트) 단위에서 시작** → OOV가 원천적으로 불가능
- **자주 나오는 쌍을 합침** → vocab을 원하는 크기로 제한 가능

GPT-2, GPT-4, LLaMA 등 최신 언어 모델이 모두 BPE 기반 토크나이저를 사용합니다.

이 노트북에서는 BPE 알고리즘의 핵심 함수인 `get_stats()` 와 `merge()` 를 직접 구현하고,  
학습 루프를 돌려보고, `BasicTokenizer` 클래스로 실제 텍스트에 적용해봅니다.

---
## 1단계: BPE의 핵심 직관

BPE의 아이디어는 단순합니다:

> **"가장 자주 붙어 나오는 글자 쌍을 하나로 합쳐서 새 토큰으로 만들자.  
> 이 과정을 원하는 vocab 크기에 도달할 때까지 반복한다."**

아래 예제로 직관을 먼저 잡아봅시다.  
`"aaabdaaabac"` 에서 가장 자주 붙어 나오는 쌍을 눈으로 찾아보세요.

In [ ]:
# 아이디어: "자주 붙어 나오는 쌍을 하나로 합치자"
text = "aaabdaaabac"

# 글자로 쪼개면
tokens = list(text)
print("초기 토큰:", tokens)
print("토큰 수:", len(tokens))

# 가장 많이 붙어 나오는 쌍은? → 눈으로 찾아보기
# 힌트: 'a'가 많이 나오는데, 'a' 뒤에 무엇이 가장 많이 따라오나요?

---
## 2단계: get_stats() 직접 구현

눈으로 세는 것은 한계가 있습니다. 코드로 **모든 인접 쌍의 빈도**를 자동으로 세는 함수를 만들어봅시다.

핵심 아이디어: `zip(ids, ids[1:])` 으로 인접한 두 원소씩 묶습니다.

```
ids    = [1, 2, 3, 1, 2]
ids[1:]= [2, 3, 1, 2]
zip    → (1,2), (2,3), (3,1), (1,2)
```

In [ ]:
def get_stats(ids):
    """인접한 쌍의 등장 횟수를 세어 딕셔너리로 반환"""
    counts = {}
    for pair in zip(ids, ids[1:]):      # zip으로 인접 쌍 생성
        counts[pair] = counts.get(pair, 0) + 1
    return counts

# 테스트
ids = [1, 2, 1, 2, 3, 1, 2]
stats = get_stats(ids)
print("쌍 빈도:", stats)
print("가장 빈번한 쌍:", max(stats, key=stats.get))

---
## 3단계: merge() 직접 구현

가장 빈번한 쌍을 찾았으면, 그 쌍을 **새로운 ID 하나**로 교체합니다.

```
병합 전: [1, 2, 3, 1, 2, 4]   pair=(1,2), new_id=10
병합 후: [10,    3, 10,    4]   (1,2) 쌍이 모두 10으로 교체
```

주의: 이미 합쳐진 원소는 건너뛰어야 합니다. 그래서 `i += 2` 를 사용합니다.

In [ ]:
def merge(ids, pair, new_id):
    """ids에서 pair를 찾아 new_id로 교체한 새 목록 반환"""
    new_ids = []
    i = 0
    while i < len(ids):
        # 현재 위치에서 pair가 시작되는지 확인
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            new_ids.append(new_id)
            i += 2   # 쌍을 한 번에 건너뜀
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

# 테스트
result = merge([1, 2, 3, 1, 2, 4], pair=(1, 2), new_id=10)
print("병합 전:", [1, 2, 3, 1, 2, 4])
print("병합 후:", result)
print("→ (1,2) 쌍이 모두 10으로 교체됨")

---
## 4단계: BPE 학습 루프 직접 돌려보기

이제 `get_stats()` 와 `merge()` 를 조합해서 BPE 학습 루프를 만들어봅시다.

학습 과정:
1. 텍스트를 UTF-8 바이트로 변환 (0~255)
2. 가장 빈번한 쌍을 찾는다
3. 그 쌍을 새 ID(256, 257, ...)로 교체한다
4. 병합 규칙을 기록한다
5. 원하는 횟수만큼 2~4를 반복한다

In [ ]:
# 텍스트를 UTF-8 바이트로 변환
text = "aaabdaaabac"
ids = list(text.encode("utf-8"))  # 각 글자의 ASCII(= UTF-8) 바이트 값
print(f"초기 ID 목록: {ids}")
print(f"(a=97, b=98, c=99, d=100)")
print(f"토큰 수: {len(ids)}")

# 3회 병합 수행
vocab_size = 256   # 기본 바이트 0~255
merges = {}        # 병합 규칙 기록: {(old1, old2): new_id}

for step in range(3):
    stats = get_stats(ids)
    best_pair = max(stats, key=stats.get)   # 가장 빈번한 쌍 선택
    new_id = vocab_size + step              # 새 ID는 256, 257, 258, ...
    ids = merge(ids, best_pair, new_id)     # 병합 적용
    merges[best_pair] = new_id             # 규칙 저장

    print(f"\n{step+1}회 병합:")
    print(f"  병합 쌍: {best_pair} (빈도: {stats[best_pair]}회)")
    print(f"  새 ID: {new_id}")
    print(f"  결과: {ids}  (길이: {len(ids)})")

print(f"\n병합 규칙: {merges}")

---
## 5단계: BPE 인코딩 — 병합 규칙을 순서대로 적용

학습이 끝나면 **병합 규칙(merges)** 이 완성됩니다.

새 텍스트를 인코딩할 때는:
1. 텍스트를 UTF-8 바이트로 변환
2. 학습된 병합 규칙을 **순서대로** 적용

순서가 중요합니다. 학습 때 먼저 병합한 쌍부터 적용해야 합니다.

In [ ]:
# 학습된 병합 규칙으로 새 텍스트 인코딩
def bpe_encode(text, merges):
    """학습된 병합 규칙을 순서대로 적용해 인코딩"""
    ids = list(text.encode("utf-8"))   # 1. 바이트로 변환
    print(f"초기 바이트: {ids}")

    # 2. 병합 규칙을 순서대로 적용
    for pair, new_id in merges.items():
        ids = merge(ids, pair, new_id)
        print(f"병합 {pair} → {new_id}: {ids}")

    return ids

result = bpe_encode("aaab", merges)
print(f"\n최종 결과: {result}")

---
## 6단계: 왜 256 바이트에서 시작하는가?

BPE는 **UTF-8 바이트** 단위에서 시작합니다. 바이트는 0~255, 총 256가지 값밖에 없습니다.

어떤 언어의 어떤 글자도 결국 1~4개의 바이트로 표현됩니다.  
따라서 256개의 기본 토큰만 있으면 **OOV가 원천적으로 불가능**합니다.

In [ ]:
# 어떤 언어도 결국 0~255 바이트로 표현됨
examples = ["A", "한", "😀", "العربية"]
for text in examples:
    encoded = list(text.encode("utf-8"))
    print(f"'{text}' → {encoded}  ({len(encoded)}바이트)")

print("\n→ 256개 기본 토큰만 있으면 OOV가 원천적으로 불가능!")

---
## 7단계: BasicTokenizer 클래스 사용

위에서 손으로 구현한 `get_stats()`, `merge()`, 학습 루프가 모두  
`basic_tokenizer.py` 의 `BasicTokenizer` 클래스에 정리되어 있습니다.

실제 텍스트로 학습해봅시다.

In [ ]:
# 클래스 import 및 실제 텍스트로 학습
import sys, os

# 현재 노트북이 있는 폴더(stage3_bpe)를 파이썬 경로에 추가
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from basic_tokenizer import BasicTokenizer, get_stats, merge

tokenizer = BasicTokenizer()

# 같은 문장을 반복해서 학습 데이터로 사용 (패턴 빈도를 높이기 위해)
train_text = """The quick brown fox jumps over the lazy dog.
The quick brown fox jumps over the lazy dog.
The quick brown fox jumps over the lazy dog."""

# vocab_size=280: 기본 256바이트 + 24회 병합
tokenizer.train(train_text, vocab_size=280, verbose=True)
print(f"\n학습 완료! vocab_size: {tokenizer.vocab_size}")
print(f"병합 규칙 수: {len(tokenizer.merges)}")

In [ ]:
# 인코딩·디코딩 왕복 테스트
test_texts = ["The quick brown fox", "hello world", "the lazy dog"]

for text in test_texts:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded)
    match = "성공" if text == decoded else "실패"
    print(f"[{match}] '{text}'")
    print(f"   인코딩: {encoded}")
    print(f"   디코딩: '{decoded}'")
    print(f"   압축: {len(text.encode())}바이트 → {len(encoded)}토큰")
    print()

In [ ]:
# 어휘집 저장 (.model + .vocab)
# .model : 병합 규칙 (인코딩에 필요)
# .vocab : 사람이 읽기 쉬운 어휘 목록 (확인용)
prefix = "basic_tokenizer"
tokenizer.save(prefix)
print(f"저장 완료: {prefix}.model, {prefix}.vocab")

# .model 파일 내용 일부 확인
with open(f"{prefix}.model", encoding="utf-8") as f:
    lines = f.readlines()
print(f"\n.model 파일 (처음 10줄):")
for line in lines[:10]:
    print(f"  {line.rstrip()}")

In [ ]:
# 불러오기 후 동일한지 확인
tok2 = BasicTokenizer()
tok2.load(f"{prefix}.model")

text = "The quick brown fox"
ids1 = tokenizer.encode(text)
ids2 = tok2.encode(text)
print(f"원본:       {ids1}")
print(f"불러온 것:  {ids2}")
print(f"동일한가? {ids1 == ids2}")

---
## 최종 정리: char / word / BPE 비교

세 단계를 모두 마쳤습니다. 지금까지 배운 내용을 정리합니다.

### 세 토크나이저 비교

| 항목 | 글자 단위 | 단어 단위 | BPE |
|------|-----------|-----------|-----|
| 기본 단위 | 글자 1개 | 공백 기준 단어 | UTF-8 바이트 |
| vocab 크기 | 약 100개 | 수십만 개 | 자유롭게 설정 (보통 30,000~100,000) |
| OOV | 거의 없음 | 자주 발생 | 불가능 |
| 토큰 효율 | 낮음 | 높음 | 중간~높음 |
| 실제 사용 | 연구/교육용 | 거의 안 씀 | GPT-2, GPT-4, LLaMA 등 |

### BPE 핵심 개념 3가지

1. **`get_stats()`**: 인접한 쌍의 빈도를 세는 함수
2. **`merge()`**: 가장 빈번한 쌍을 새 ID로 교체하는 함수
3. **학습 루프**: 원하는 vocab_size에 도달할 때까지 get_stats + merge를 반복

---

## 다음 단계 예고: RegexTokenizer

GPT-2가 실제로 사용하는 토크나이저는 여기서 한 단계 더 나아갑니다.

**RegexTokenizer** 는 BPE를 적용하기 전에 **정규식(Regex)** 으로 텍스트를 미리 분리합니다.  
예를 들어 `"don't"` 를 `"don"` 과 `"'t"` 로 먼저 나눈 뒤 BPE를 적용합니다.

이렇게 하면 단어 경계를 무시한 비직관적인 병합(예: `" the"` + `" cat"` = `" thecat"`)을 방지할 수 있습니다.